# Grid Transformations
In the context of modern data engineering and Databricks, "Grid transformations" typically refer to three distinct concepts: reshaping tabular data (Pivot/Unpivot), Spatial Grid discretization (H3), and ML Hyperparameter Grid Search.

Given our interest in the Databricks Lakehouse architecture, I will break down how to implement these transformations with high code quality.





In [0]:
spark.sql("USE CATALOG databricks_simulated_retail_customer_data")
spark.sql("USE v01")

## 1. Reshaping Transformations (Pivot & Unpivot)
This is the most common interpretation: changing the structure of our data "grid" to make it suitable for either BI tools (which often prefer long formats) or ML features (which often prefer wide formats).

### Pivot (Long to Wide)
In Spark, pivoting allows you to turn unique values from one column into multiple specific columns.

In [0]:
from pyspark.sql import functions as F

df = spark.table("sales")

# Example: Converting monthly sales rows into columns
df = df.withColumn("month", F.month("order_date")).withColumn("year", F.year("order_date"))

# Format month as "number of month_month_sales" before pivoting
df = df.withColumn("month", F.concat(F.col("month").cast("string"), F.lit("_month_sales")))

df_pivoted = (df.groupBy("product_category", "year")
              .pivot("month")
              .agg(F.sum("total_price"))
              .fillna(0)) # Always handle nulls after a pivot

df_pivoted.display()

### Unpivot (Wide to Long)
Standard Spark SQL uses the stack function to "melt" a wide grid back into a narrow format, which is often more efficient for the Silver and Gold layers described in our Medallion architecture.

In [0]:
df_pivoted.createOrReplaceTempView("pivot_table")

In [0]:
%sql
-- Reshaping 2, 8, 9, 10 columns into a single 'month' column
SELECT 
  product_category, year, 
  stack(4, "2", `2_month_sales`, "8", `8_month_sales`, "9", `9_month_sales`, "10", `10_month_sales`) AS (month, sales)
FROM pivot_table

## 2. Spatial Grid Transformations (H3 Indexing)
Databricks has integrated H3 (Hexagonal Hierarchical Geospatial Indexing) as a first-class citizen. This transforms precise Latitude/Longitude coordinates into a discrete hexagonal grid. This is critical for performance because it allows you to perform "joins" on spatial data using simple string/long comparisons rather than expensive distance calculations.

Why this matters for our architecture: It enables consistent spatial analysis across BI and ML.

In [0]:
spark.sql("USE CATALOG kindred_sample_kindred_maid_based_real_time_location_data_geolocation_mobility_data_34_geos")
spark.sql("USE kindred_15d9b79b_b598_4a93_af23_19d59d02c75e_samples")

In [0]:
%sql
-- Transforming coordinate points into an H3 Grid Cell at resolution 8
SELECT 
  h3_longlatash3(longitude, latitude, 8) AS h3_cell,
  count(*) as activity
FROM location_data_sample_data_b7b9a003798a40bd9e91f61219a35e1c
GROUP BY 1

## 3. Window Grid Transformations
Window functions allow you to transform data based on a "grid" or frame of surrounding rows without collapsing them into a single grouping.

**Example**: If you have sales data ordered by date, each row gets a moving_avg column showing the average price over the last 11 transactions for that product category, useful for smoothing out trends and detecting anomalies.

Why it's called a "grid transformation": The window frame (rowsBetween) creates a moving rectangular "grid" that slides down the dataset, recalculating the metric at each position.

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Defining a grid frame: Partition by category, ordered by time
window_spec = Window.partitionBy("product_category").orderBy("order_date")

# Calculating a moving average over a grid of rows
# specifies a frame of 11 rows — the current row (0) plus the 10 preceding rows (-10). 
# For each row, it computes the average total_price across those 11 rows.
df_with_moving_avg = df.withColumn(
    "moving_avg", 
    F.avg("total_price").over(window_spec.rowsBetween(-10, 0))
)

# Preserves all rows: 
# Unlike groupBy, which collapses data into summary rows, 
# window functions add the calculated metric as a new column while keeping every original row.
df_with_moving_avg.display()

## 4. ML Hyperparameter Grid Search
In the context of the AI/ML-Ready platform mentioned in our architecture, "Grid" refers to GridSearch. This is a transformation process where you systematically work through a grid of hyperparameter combinations to find the optimal model.

On Databricks, we use Hyperopt or Scikit-learn with SparkTrials to distribute this grid transformation across the cluster.

In [0]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the grid of parameters
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

# In a Databricks environment, use Spark-backed Joblib to distribute this grid
grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, n_jobs=-1)
# grid_search.fit(X_train, y_train)

## Summary for your Strategy
If you are designing the Silver/Gold layers:
* Pivot/Unpivot should happen in the Silver to Gold transition to prepare data for the Semantic Layer.
* H3 Spatial Grids should be computed in the Silver layer to enable high-performance analytics.
* Windowing is essential for Data Quality checks (detecting spikes/anomalies) in the transformation layer.